[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day09-prefill-vs-decode-arithmetic-intensity.ipynb)

# Day 9 — Prefill vs Decode; Arithmetic Intensity

Today you measure the two-phase split yourself: prefill is compute-bound (wide parallel GEMMs, intensity ~2000 FLOP/byte), decode is memory-bandwidth-bound (~1 FLOP/byte), and batching amortizes the weight stream. CPU-OK; T4 gives the cleanest timing numbers.

In [ ]:
# One and only pip install cell
!pip install -q transformers torch --index-url https://download.pytorch.org/whl/cpu
# (On a T4 runtime, replace with: !pip install -q transformers torch  )

## 1. Setup: load SmolLM2-135M

In [ ]:
import time, math
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
tok = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-135M')
model = AutoModelForCausalLM.from_pretrained('HuggingFaceTB/SmolLM2-135M').to(device).eval()
params = sum(p.numel() for p in model.parameters())
print(f'params: {params/1e6:.1f}M')
# Expected output:
# Device: cpu            (or cuda on a T4)
# params: 134.5M

## 2. Time prefill: one parallel pass over the prompt

Prefill processes every prompt token in one forward pass. Measure median wall time over 5 runs for prompt lengths 128 / 512 / 2048.

In [ ]:
def time_prefill(n_tokens, repeats=5):
    ids = torch.randint(0, tok.vocab_size, (1, n_tokens)).to(device)
    with torch.no_grad():
        model(ids)  # warmup
    ts = []
    with torch.no_grad():
        for _ in range(repeats):
            t0 = time.perf_counter(); model(ids); ts.append(time.perf_counter() - t0)
    return min(ts)  # min ~ median for timing; robust to jitter

for n in (128, 512, 2048):
    s = time_prefill(n)
    print(f'prompt {n:5d} tok: prefill {s*1000:8.2f} ms total  -> {s/n*1000:6.3f} ms/token')
# Expected (T4): per-token time roughly FLAT as length grows (parallel) --
#   e.g. 0.05-0.3 ms/token. On CPU: slower but still flat-ish.

## 3. Time decode: one token at a time, KV cache on

`model.generate` uses the KV cache by default (Day 8). Generate 100 tokens, subtract prefill, divide by 100.

In [ ]:
prompt = 'The transformer architecture changed machine learning because'
ids = tok(prompt, return_tensors='pt').input_ids.to(device)
with torch.no_grad():
    pre = time_prefill(ids.shape[1])
    t0 = time.perf_counter()
    out = model.generate(ids, max_new_tokens=100, do_sample=False, pad_token_id=tok.eos_token_id)
    total = time.perf_counter() - t0
decode_ms_per_tok = (total - pre) / 100 * 1000
print(f'prefill: {pre*1000:.1f} ms | decode+100 tok: {total*1000:.0f} ms | decode {decode_ms_per_tok:.2f} ms/token')
print('generated:', tok.decode(out[0], skip_special_tokens=True)[:120], '...')
# Expected (T4): decode ~2-8 ms/token -- 10-50x slower PER TOKEN than prefill.
# Same ratio on CPU, both slower. This gap is the whole day in one number.

## 4. Roofline: ridge points and the three workloads

Plot the roofline for T4 / A100 / H100 and place: prefill-8B-2k, decode-8B-b1, decode-8B-b32.

In [ ]:
import matplotlib.pyplot as plt

# peak fp16 TFLOP/s, HBM TB/s (decimal, spec values)
gpus = {'T4': (65, 0.32), 'A100-80GB': (312, 2.0), 'H100': (989, 3.35)}
for g, (pi, bw) in gpus.items():
    print(f'{g:10s} ridge = {pi*1e12/(bw*1e12):6.1f} FLOP/byte')
# Expected:
# T4         ridge =  203.1 FLOP/byte
# A100-80GB  ridge =  156.0 FLOP/byte
# H100       ridge =  295.2 FLOP/byte

pi, bw = [x*1e12 for x in gpus['H100']]  # roofline on H100
I = [10**e for e in [i/20 for i in range(-20, 81)]]  # 0.1 .. 1e4 FLOP/byte
perf = [min(pi, bw*i)/1e12 for i in I]  # TFLOP/s
plt.figure(figsize=(7, 5))
plt.loglog(I, perf, 'k-', lw=2, label='H100 roofline')
points = {'prefill 8B 2k ctx (I~2000)': (2000, min(pi, bw*2000)/1e12),
          'decode 8B b1 (I~1)': (1.0, min(pi, bw*1.0)/1e12),
          'decode 8B b32 (I~32)': (32.0, min(pi, bw*32.0)/1e12)}
for name, (x, y) in points.items():
    plt.loglog([x], [y], 'o', ms=9); plt.annotate(name, (x, y), xytext=(6, 6), textcoords='offset points')
plt.axvline(pi/bw, ls='--', c='gray'); plt.annotate('ridge ~295', (pi/bw, 30), rotation=90)
plt.xlabel('Arithmetic intensity (FLOP/byte)'); plt.ylabel('Attainable perf (TFLOP/s)')
plt.title('Roofline: prefill rides the compute roof, decode hugs the bandwidth roof')
plt.legend(); plt.grid(True, which='both', alpha=0.3); plt.show()

## 5. Close the loop: predicted vs measured decode ms/token

Prediction: ms/token = model-bytes / bandwidth. For SmolLM2-135M in fp32 on CPU this is a rough bound; on a T4 (fp16) it should land within 2x. The gap = kernel launch overhead + attention traffic -- note it, don't hand-wave it.

In [ ]:
dtype_bytes = next(model.parameters()).element_size()
model_bytes = params * dtype_bytes
bw = {'cpu': 20e9, 'cuda': 0.32e12}[device]  # ~20 GB/s typical CPU DRAM; T4 HBM 0.32 TB/s
pred_ms = model_bytes / bw * 1000
print(f'model bytes: {model_bytes/1e9:.2f} GB | assumed BW: {bw/1e9:.0f} GB/s')
print(f'predicted decode: {pred_ms:.2f} ms/token | measured: {decode_ms_per_tok:.2f} ms/token')
print(f'ratio measured/predicted: {decode_ms_per_tok/pred_ms:.1f}x  (target: within 2x on GPU)')
# Expected on T4: ratio ~1.5-3x. On CPU the crude 20 GB/s assumption makes
# the bound looser -- the point is the ORDER of magnitude, not the digit.

## 6. Checkpoints (answer cold before moving on)

1. H100: 989 TFLOP/s, 3.35 TB/s. Ridge point? -> ~295 FLOP/byte.
2. Decode intensity for 8B fp16, batch 1? -> ~1 FLOP/byte: 295x below the ridge, memory-bandwidth-bound.
3. Why does batching raise decode throughput? -> One 16 GB weight read serves 32 tokens' worth of work at batch 32; aggregate tok/s rises 32x while still memory-bound.
4. 8B/H100: 2k-token prefill ~0.1 s, 200 decode tokens ~1 s. Which dominates, and the SLO terms? -> Decode (~95%). TTFT covers prefill; TPOT covers steady-state decode.